In [1]:
import nltk
from nltk.util import ngrams
from collections import Counter
import re

# ------------------------------------------------
# Sample Corpus (training text)
# ------------------------------------------------
corpus = """
Natural language processing is a field of artificial intelligence.
Machine learning and natural language processing are related.
Language models predict probabilities of words.
Spelling correction improves text quality.
"""

# ------------------------------------------------
# Preprocess corpus
# ------------------------------------------------
words = re.findall(r'\w+', corpus.lower())

# Word frequency
word_freq = Counter(words)

# Bigram frequencies
bigrams = list(ngrams(words, 2))
bigram_freq = Counter(bigrams)

vocab = set(words)

# ------------------------------------------------
# Levenshtein Edit Distance
# ------------------------------------------------
def edit_distance(word1, word2):

    m = len(word1)
    n = len(word2)

    dp = [[0]*(n+1) for _ in range(m+1)]

    for i in range(m+1):
        dp[i][0] = i

    for j in range(n+1):
        dp[0][j] = j

    for i in range(1,m+1):
        for j in range(1,n+1):

            cost = 0 if word1[i-1] == word2[j-1] else 1

            dp[i][j] = min(
                dp[i-1][j]+1,      # deletion
                dp[i][j-1]+1,      # insertion
                dp[i-1][j-1]+cost  # substitution
            )

    return dp[m][n]


# ------------------------------------------------
# Candidate generation
# ------------------------------------------------
def candidates(word):

    possible = []

    for vocab_word in vocab:

        dist = edit_distance(word, vocab_word)

        if dist <= 2:
            possible.append(vocab_word)

    return possible


# ------------------------------------------------
# Bigram probability
# P(current | previous)
# ------------------------------------------------
def bigram_probability(prev_word,current_word):

    numerator = bigram_freq[(prev_word,current_word)] + 1
    denominator = word_freq[prev_word] + len(vocab)

    return numerator/denominator


# ------------------------------------------------
# Spell correction
# ------------------------------------------------
def correct_sentence(sentence):

    tokens = sentence.lower().split()

    corrected = []

    for i,word in enumerate(tokens):

        if word in vocab:
            corrected.append(word)
            continue

        cand = candidates(word)

        if not cand:
            corrected.append(word)
            continue

        best_word = word
        highest_prob = 0

        prev_word = corrected[i-1] if i>0 else ""

        for c in cand:

            prob = bigram_probability(prev_word,c)

            if prob > highest_prob:
                highest_prob = prob
                best_word = c

        corrected.append(best_word)

    return " ".join(corrected)


# ------------------------------------------------
# Test
# ------------------------------------------------
sentence = input("Enter sentence: ")

corrected = correct_sentence(sentence)

print("Corrected Sentence:")
print(corrected)


Enter sentence: natural language processing
Corrected Sentence:
natural language processing
